In [8]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [9]:
SEED = 42
np.random.seed(SEED)


In [10]:
# =========================================================
# 1) TEMPERATURE REGRESSION
# =========================================================
exp_temp_encoding = Experiment(
    name="temperature_regression_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="temperature",
        target_mode="regression",
        # target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=[
            ("temperature",),
            # ("temperature", "humidity"),
            # ("temperature", "humidity", "pressure"),
            # ("temperature", "humidity", "pressure", "wind_speed"),
            # ("temperature", "humidity", "pressure", "wind_speed", "wind_direction"),
        ],
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=("Vancouver",),
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="regression",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers = [
            (16,),
            (32,),
            (64,),
            (32, 64),
            (64, 32),
            (64, 64),
            (128, 64),
            (128, 64, 32),
            (64, 64, 32),
            (16, 32),
            (32, 64),
            (64, 128),
            (16, 32, 64),
            (32, 64, 128),
            (32, 64, 64),
            (64, 32, 64),
            (128, 64, 128),
            (32, 64, 32),
            (64, 128, 64),
            (32, 64, 128, 64),
        ],
        loss="huber",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

# =========================================================
# 2) WIND (>=6 m/s) BINARY CLASSIFICATION
# =========================================================
exp_wind_encoding = Experiment(
    name="wind6_binary_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="wind_speed",
        target_mode="binary",
        target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=[
            ("wind_speed",),
            # ("wind_speed", "wind_direction"),
            # ("wind_speed", "wind_direction", "pressure"),
            # ("wind_speed", "wind_direction", "pressure", "humidity"),
            # ("wind_speed", "wind_direction", "pressure", "humidity", "temperature"),
        ],
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=("Vancouver",),
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="binary",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers = [
            (16,),
            (32,),
            (64,),
            (32, 64),
            (64, 32),
            (64, 64),
            (128, 64),
            (128, 64, 32),
            (64, 64, 32),
            (16, 32),
            (32, 64),
            (64, 128),
            (16, 32, 64),
            (32, 64, 128),
            (32, 64, 64),
            (64, 32, 64),
            (128, 64, 128),
            (32, 64, 32),
            (64, 128, 64),
            (32, 64, 128, 64),
        ],
        loss="binary_cross_entropy",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

experiments = [exp_temp_encoding, exp_wind_encoding]


In [11]:
search = Search()

results1 = search.run(exp_temp_encoding)



Starting experiment: temperature_regression_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 681.26it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 743.43it/s]



Configuration run 1/20:
WEATHER (variable):
  - input_variables: ('temperature',)
MLP (variable):
  - hidden_layers: (16,)

Training model


...

Early stopping at epoch 64, best val_loss=3.779669 after 50 epochs without improvement.
Training finished in 1.82 seconds

Configuration run 19/20:
WEATHER (variable):
  - input_variables: ('temperature',)
MLP (variable):
  - hidden_layers: (64, 128, 64)

Training model


Training:  25%|██▌       | 101/400 [00:04<00:14, 21.26it/s, acc=n/a, loss=1.4337, lr=0.00362372]


Early stopping at epoch 102, best val_loss=0.994693 after 50 epochs without improvement.
Training finished in 4.75 seconds

Configuration run 20/20:
WEATHER (variable):
  - input_variables: ('temperature',)
MLP (variable):
  - hidden_layers: (32, 64, 128, 64)

Training model


Training:  38%|███▊      | 154/400 [00:08<00:13, 18.42it/s, acc=n/a, loss=4.2028, lr=0.00212726]

Early stopping at epoch 155, best val_loss=3.715424 after 50 epochs without improvement.
Training finished in 8.37 seconds

Experiment finished | total runs = 20



In [15]:
from IPython.core.display import HTML

for run in results1:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.65:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.60:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.58:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.55:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"AUC      : {metrics['auc']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {auc_color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>AUC</b>: {auc_val:.4f}
            </div>
            """
        ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        # display(HTML(
        #     f"""
        #     <div style="
        #         font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
        #         font-size: 13px;
        #         color: {color};
        #         padding-left: 12px;
        #         margin: 4px 0;
        #     ">
        #         <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
        #     </div>
        #     """
        # ))
        print(f"Accuracy |err|≤2°C   : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
  MLP:
    - hidden_layers: (16,)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 4.5272
MSE              : 38.7064
RMSE             : 6.2214
Accuracy |err|≤2°C   : 0.3201


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
  MLP:
    - hidden_layers: (32,)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 3.3028
MSE              : 17.2558
RMSE             : 4.1540
Accuracy |err|≤2°C   : 0.3628


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
  MLP:
    - hidden_layers: (64,)
---------

In [13]:
search = Search()

results2 = search.run(exp_wind_encoding)


Starting experiment: wind6_binary_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:01<00:00, 762.82it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 837.48it/s]



Configuration run 1/20:
WEATHER (variable):
  - input_variables: ('wind_speed',)
MLP (variable):
  - hidden_layers: (16,)

Training model


...

Early stopping at epoch 54, best val_loss=0.680456, train_acc=0.6640, val_acc=0.5855 after 50 epochs without improvement.
Training finished in 2.61 seconds

Configuration run 19/20:
WEATHER (variable):
  - input_variables: ('wind_speed',)
MLP (variable):
  - hidden_layers: (64, 128, 64)

Training model


Training:  14%|█▍        | 55/400 [00:05<00:32, 10.60it/s, acc=0.7006, loss=0.5878, lr=0.00575355]


Early stopping at epoch 56, best val_loss=0.659882, train_acc=0.7006, val_acc=0.6118 after 50 epochs without improvement.
Training finished in 5.20 seconds

Configuration run 20/20:
WEATHER (variable):
  - input_variables: ('wind_speed',)
MLP (variable):
  - hidden_layers: (32, 64, 128, 64)

Training model


Training:  13%|█▎        | 51/400 [00:05<00:38,  9.04it/s, acc=0.6786, loss=0.5930, lr=0.00598956]


Early stopping at epoch 52, best val_loss=0.657740, train_acc=0.6786, val_acc=0.5526 after 50 epochs without improvement.
Training finished in 5.65 seconds

Experiment finished | total runs = 20



In [16]:
from IPython.core.display import HTML

for run in results2:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.60:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.55:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.53:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.51:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"Auc      : {metrics['auc']:.4f}")
        # display(HTML(
        #     f"""
        #     <div style="
        #         font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
        #         font-size: 13px;
        #         color: {auc_color};
        #         padding-left: 12px;
        #         margin: 4px 0;
        #     ">
        #         <b>AUC</b>: {auc_val:.4f}
        #     </div>
        #     """
        # ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
            </div>
            """
        ))
        print(f"Accuracy |err|≤2°C : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('wind_speed',)
  MLP:
    - hidden_layers: (16,)
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5335
Precision: 0.6043
Recall   : 0.5885
Auc      : 0.5214


EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('wind_speed',)
  MLP:
    - hidden_layers: (32,)
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.4970
Precision: 0.5944
Recall   : 0.4427
Auc      : 0.5188


EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('wind_speed',)
  MLP:
    - hidden_layers: (64,)
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY 